In [1]:
config = """
seed: 42
language: eng

data_root: data/dev_phase/subtask1
train_file: train/eng.csv
dev_file: dev/eng.csv

text_col: text
label_col: polarization
id_col: id

num_labels: 2
model_name: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
max_length: 256

test_size: 0.1
balance_test: false

use_class_weights: false

#learning_rate: 1e-5
learning_rate: 5e-6
lr_scheduler_type: cosine

optim: adamw_torch
max_grad_norm: 1


weight_decay: 0.01
warmup_ratio: 0.10
label_smoothing_factor: 0.05


num_train_epochs: 5

per_device_train_batch_size: 4
per_device_eval_batch_size: 8
gradient_accumulation_steps: 4

fp16: false

eval_strategy: steps
eval_steps: 100

save_strategy: steps
save_steps: 100
save_total_limit: 1

load_best_model_at_end: true

metric_for_best_model: eval_macro_f1
greater_is_better: true

logging_steps: 50
report_to: wandb

output_dir: /kaggle/working/outputs
outputs_dir: /kaggle/working/outputs
predictions_file: /kaggle/working/outputs/pred_eng.csv
confusion_matrix_file: /kaggle/working/outputs/confusion_matrix_eng.png


cv_folds: 5
cv_start_fold: 1
cv_end_fold: 5
resume_from_checkpoint: false
"""

In [2]:
with open("config.yaml", "w") as f:
    f.write(config)

print("Saved config.yaml")

Saved config.yaml


In [3]:
import yaml

def read_cfg(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)
        
cfg = read_cfg()


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import AutoModel, AutoTokenizer

class RobertaMpnetFusion(nn.Module):
    def __init__(
        self,
        roberta,
        mpnet,
        num_labels: int = 2,
        mpnet_proj_dim: int = 256,
        dropout: float = 0.3,
        freeze_mpnet: bool = True,
    ):
        super().__init__()
        self.roberta = roberta
        self.mpnet = mpnet
        

        if freeze_mpnet:
            for p in self.mpnet.parameters():
                p.requires_grad = False

        roberta_dim = self.roberta.config.hidden_size
        self.config = self.roberta.config
        mpnet_dim = self.mpnet.config.hidden_size

        self.mpnet_proj = nn.Linear(mpnet_dim, mpnet_proj_dim)

        self.classifier = nn.Sequential(
            nn.Linear(roberta_dim + mpnet_proj_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_labels),
        )

        self.num_labels = num_labels
        self.loss_fn = nn.CrossEntropyLoss()

    @staticmethod
    def mean_pool(last_hidden, attention_mask):
        # mean pooling with mask
        mask = attention_mask.unsqueeze(-1).float()
        return (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(
        self,
        roberta_input_ids=None,
        roberta_attention_mask=None,
        mpnet_input_ids=None,
        mpnet_attention_mask=None,
        labels=None,
        **kwargs,
    ):
        # RoBERTa CLS
        rob_out = self.roberta(
            input_ids=roberta_input_ids,
            attention_mask=roberta_attention_mask,
        )
        rob_cls = rob_out.last_hidden_state[:, 0]  

        # MPNet pooled embedding
        mp_out = self.mpnet(
            input_ids=mpnet_input_ids,
            attention_mask=mpnet_attention_mask,
        )
        mp_emb = self.mean_pool(mp_out.last_hidden_state, mpnet_attention_mask)
        mp_emb = F.normalize(mp_emb, p=2, dim=1)
        mp_emb = self.mpnet_proj(mp_emb)

        fused = torch.cat([rob_cls, mp_emb], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            labels = labels.long()
            loss = self.loss_fn(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)



def get_model_and_tokenizer(model_name: str, num_labels: int):
    roberta_tok = AutoTokenizer.from_pretrained("roberta-large", use_fast=True)
    mpnet_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2", use_fast=True)

    roberta = AutoModel.from_pretrained("roberta-large")
    mpnet = AutoModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

    model = RobertaMpnetFusion(
        roberta=roberta,
        mpnet=mpnet,
        num_labels=num_labels,
        freeze_mpnet=True,
    )
    return model, roberta_tok, mpnet_tok


In [5]:
import torch
from torch.utils.data import Dataset


class FusionTextDataset(torch.utils.data.Dataset):
    def __init__(self, df, roberta_tok, mpnet_tok, text_col="text", label_col=None, max_length=256):
        self.df = df.reset_index(drop=True)
        self.roberta_tok = roberta_tok
        self.mpnet_tok = mpnet_tok
        self.text_col = text_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx][self.text_col])

        r = self.roberta_tok(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        m = self.mpnet_tok(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        item = {
            "roberta_input_ids": torch.tensor(r["input_ids"], dtype=torch.long),
            "roberta_attention_mask": torch.tensor(r["attention_mask"], dtype=torch.long),
            "mpnet_input_ids": torch.tensor(m["input_ids"], dtype=torch.long),
            "mpnet_attention_mask": torch.tensor(m["attention_mask"], dtype=torch.long),
        }

        if self.label_col is not None:
            item["labels"] = torch.tensor(int(self.df.iloc[idx][self.label_col]), dtype=torch.long)

        return item





In [6]:
from transformers import DataCollatorWithPadding

class FusionCollator:
    def __init__(self, roberta_tok, mpnet_tok):
        self.r_collate = DataCollatorWithPadding(tokenizer=roberta_tok, padding=True)
        self.m_collate = DataCollatorWithPadding(tokenizer=mpnet_tok, padding=True)

    def __call__(self, features):
        r_feats = [{"input_ids": f["roberta_input_ids"], "attention_mask": f["roberta_attention_mask"]} for f in features]
        m_feats = [{"input_ids": f["mpnet_input_ids"], "attention_mask": f["mpnet_attention_mask"]} for f in features]

        r_batch = self.r_collate(r_feats)
        m_batch = self.m_collate(m_feats)

        batch = {
            "roberta_input_ids": r_batch["input_ids"],
            "roberta_attention_mask": r_batch["attention_mask"],
            "mpnet_input_ids": m_batch["input_ids"],
            "mpnet_attention_mask": m_batch["attention_mask"],
        }

        if "labels" in features[0]:
            batch["labels"] = torch.stack([f["labels"] for f in features])

        return batch


2026-01-09 14:19:28.430539: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767968368.645128      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767968368.703664      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767968369.217052      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767968369.217098      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767968369.217101      24 computation_placer.cc:177] computation placer alr

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

def make_compute_metrics(confusion_matrix_path: str, print_in_terminal: bool = True):
    os.makedirs(os.path.dirname(confusion_matrix_path), exist_ok=True)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        acc = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average="macro")

        pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1], average=None)
        cm = confusion_matrix(labels, preds, labels=[0, 1])

        if print_in_terminal:
            print("\n=== Confusion Matrix (rows=true, cols=pred) ===")
            print(cm)
            print("\n=== Classification report ===")
            print(classification_report(labels, preds, digits=4))

        plt.figure(figsize=(4, 3))
        plt.imshow(cm)
        plt.xticks([0, 1], ["0", "1"])
        plt.yticks([0, 1], ["0", "1"])
        for i in range(2):
            for j in range(2):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(confusion_matrix_path)
        plt.close()

        return {
            "accuracy": acc,
            "macro_f1": macro_f1,
            "precision_0": pr[0],
            "recall_0": rc[0],
            "f1_0": f1[0],
            "precision_1": pr[1],
            "recall_1": rc[1],
            "f1_1": f1[1],
        }

    return compute_metrics


In [8]:
import numpy as np
import torch
from transformers import Trainer


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        # Forward everything except labels
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits

        if labels is not None:
            labels = labels.long()
            if self.class_weights is not None:
                w = torch.tensor(
                    self.class_weights,
                    device=logits.device,
                    dtype=torch.float,
                )
                loss_fct = torch.nn.CrossEntropyLoss(weight=w)
            else:
                loss_fct = torch.nn.CrossEntropyLoss()

            loss = loss_fct(logits, labels)
        else:
            loss = outputs.loss

        return (loss, outputs) if return_outputs else loss



def compute_class_weights(y, num_labels=2):
    y = np.asarray(y)
    counts = np.bincount(y, minlength=num_labels).astype(float)
    counts[counts == 0] = 1.0
    inv = 1.0 / counts
    weights = inv / inv.sum() * num_labels
    return weights.tolist()


In [9]:
import os
import yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import TrainingArguments, DataCollatorWithPadding, set_seed
from sklearn.model_selection import train_test_split

from pathlib import Path

import wandb
from sklearn.model_selection import StratifiedKFold





def balanced_train_test_split(df, label_col, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    counts = df[label_col].value_counts()
    if len(counts) < 2:
        return train_test_split(df, test_size=test_size, random_state=seed)

    n_test = int(round(len(df) * test_size))
    n_each = n_test // 2

    df0 = df[df[label_col] == 0]
    df1 = df[df[label_col] == 1]

    n_each = min(n_each, len(df0), len(df1))
    if n_each == 0:
        return train_test_split(df, test_size=test_size, random_state=seed, stratify=df[label_col])

    test0_idx = rng.choice(df0.index.to_numpy(), size=n_each, replace=False)
    test1_idx = rng.choice(df1.index.to_numpy(), size=n_each, replace=False)
    test_idx = np.concatenate([test0_idx, test1_idx])

    test_df = df.loc[test_idx].sample(frac=1, random_state=seed).reset_index(drop=True)
    train_df = df.drop(index=test_idx).reset_index(drop=True)
    return train_df, test_df


def read_cfg(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def cast_cfg(cfg):
    cfg["learning_rate"] = float(cfg["learning_rate"])
    cfg["weight_decay"] = float(cfg["weight_decay"])
    cfg["warmup_ratio"] = float(cfg["warmup_ratio"])
    cfg["num_train_epochs"] = int(cfg["num_train_epochs"])
    cfg["per_device_train_batch_size"] = int(cfg["per_device_train_batch_size"])
    cfg["per_device_eval_batch_size"] = int(cfg["per_device_eval_batch_size"])
    cfg["gradient_accumulation_steps"] = int(cfg.get("gradient_accumulation_steps", 1))
    cfg["max_grad_norm"] = float(cfg.get("max_grad_norm", 1.0))
    cfg["label_smoothing_factor"] = float(cfg.get("label_smoothing_factor", 0.0))
    cfg["eval_steps"] = int(cfg.get("eval_steps", 100))
    cfg["save_steps"] = int(cfg.get("save_steps", 100))
    cfg["save_total_limit"] = int(cfg.get("save_total_limit", 1))
    cfg["cv_folds"] = int(cfg.get("cv_folds", 5))
    cfg["cv_start_fold"] = int(cfg.get("cv_start_fold", 1))
    cfg["cv_end_fold"] = int(cfg.get("cv_end_fold", cfg["cv_folds"]))
    cfg["seed"] = int(cfg["seed"])
    cfg["resume_from_checkpoint"] = bool(cfg.get("resume_from_checkpoint", False))
    return cfg


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def make_args(cfg, output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy=cfg["eval_strategy"],
        eval_steps=cfg["eval_steps"] if cfg["eval_strategy"] == "steps" else None,
        save_strategy=cfg["save_strategy"],
        save_steps=cfg["save_steps"] if cfg["save_strategy"] == "steps" else None,
        save_total_limit=cfg["save_total_limit"],
        logging_strategy="steps",
        logging_steps=int(cfg["logging_steps"]),
        learning_rate=cfg["learning_rate"],
        lr_scheduler_type=cfg.get("lr_scheduler_type", "cosine"),
        optim=cfg.get("optim", "adamw_torch"),
        max_grad_norm=cfg.get("max_grad_norm", 1.0),
        label_smoothing_factor=cfg.get("label_smoothing_factor", 0.0),
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        num_train_epochs=cfg["num_train_epochs"],
        weight_decay=cfg["weight_decay"],
        warmup_ratio=cfg["warmup_ratio"],
        load_best_model_at_end=bool(cfg.get("load_best_model_at_end", True)),
        metric_for_best_model=cfg.get("metric_for_best_model", "eval_macro_f1"),
        greater_is_better=bool(cfg.get("greater_is_better", True)),
        fp16=bool(cfg.get("fp16", False) and torch.cuda.is_available()),
        report_to="wandb",
        seed=cfg["seed"],
        remove_unused_columns=False,
        save_only_model=True
    )



def main():
    cfg = read_cfg()
    wandb.init(mode="disabled")
    
    # wandb.init( project="polarization", name="fused2", config=cfg)

    cfg["learning_rate"] = float(cfg["learning_rate"])
    cfg["weight_decay"] = float(cfg["weight_decay"])
    cfg["warmup_ratio"] = float(cfg["warmup_ratio"])
    cfg["num_train_epochs"] = int(cfg["num_train_epochs"])
    cfg["per_device_train_batch_size"] = int(cfg["per_device_train_batch_size"])
    cfg["per_device_eval_batch_size"] = int(cfg["per_device_eval_batch_size"])
    cfg["gradient_accumulation_steps"] = int(cfg.get("gradient_accumulation_steps", 1))

    set_seed(cfg["seed"])
    os.makedirs(cfg["outputs_dir"], exist_ok=True)
    os.makedirs(cfg["output_dir"], exist_ok=True)

    train_path = "/kaggle/input/augment/eng_aug.csv"
    dev_path = "/kaggle/input/subtask1/dev/eng.csv"
    train_path = str(train_path)
    dev_path = str(dev_path)

    print(train_path)
    df = pd.read_csv(train_path)
    df = df[[cfg["id_col"], cfg["text_col"], cfg["label_col"]]].dropna()
    df[cfg["label_col"]] = df[cfg["label_col"]].astype(int)
    df[cfg["text_col"]] = df[cfg["text_col"]].astype(str)

    if cfg.get("balance_test", True):
        train_df, test_df = balanced_train_test_split(
            df, label_col=cfg["label_col"], test_size=cfg["test_size"], seed=cfg["seed"]
        )
    else:
        train_df, test_df = train_test_split(
            df, test_size=cfg["test_size"], random_state=cfg["seed"], stratify=df[cfg["label_col"]]
        )

    print("Train label counts:\n", train_df[cfg["label_col"]].value_counts())
    print("Test label counts:\n", test_df[cfg["label_col"]].value_counts())

    dev_df = pd.read_csv(dev_path)
    dev_df = dev_df[[cfg["id_col"], cfg["text_col"]]].dropna()
    dev_df[cfg["text_col"]] = dev_df[cfg["text_col"]].astype(str)

    skf = StratifiedKFold(n_splits=cfg["cv_folds"], shuffle=True, random_state=cfg["seed"])

    fold_scores = []
    fold_metrics_rows = []
    dev_probs_sum = np.zeros((len(dev_df), cfg["num_labels"]), dtype=np.float32)
    completed_folds = 0

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df[cfg["label_col"]].values), start=1):
        if fold < cfg["cv_start_fold"] or fold > cfg["cv_end_fold"]:
            continue

        fold_seed = cfg["seed"] + fold
        set_seed(fold_seed)

        train_df = df.iloc[tr_idx].reset_index(drop=True)
        val_df = df.iloc[va_idx].reset_index(drop=True)

        fold_out = Path(cfg["outputs_dir"]) / f"cv_fold_{fold}"
        fold_out.mkdir(parents=True, exist_ok=True)

        model, rob_tokenizer, mpnet_tok = get_model_and_tokenizer(cfg["model_name"], cfg["num_labels"])
        collator = FusionCollator(rob_tokenizer, mpnet_tok)

        train_ds = FusionTextDataset(
        train_df, rob_tokenizer, mpnet_tok,
        text_col=cfg["text_col"],
        label_col=cfg["label_col"],
        max_length=cfg["max_length"],
        )
        test_ds = FusionTextDataset(
        test_df, rob_tokenizer, mpnet_tok,
        text_col=cfg["text_col"],
        label_col=cfg["label_col"],
        max_length=cfg["max_length"],
        )


        class_weights = None
        if cfg.get("use_class_weights", True):
            class_weights = compute_class_weights(
                train_df[cfg["label_col"]].values,
                num_labels=cfg["num_labels"],
            )

        cm_path = str(Path(cfg["outputs_dir"]) / f"confusion_matrix_fold_{fold}.png")
        compute_metrics = make_compute_metrics(cm_path)

        args = make_args(cfg, fold_out)

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=test_ds,
            data_collator=collator,
            compute_metrics=compute_metrics,
            class_weights=class_weights,
        )

        if cfg.get("resume_from_checkpoint", False):
            trainer.train(resume_from_checkpoint=True)
        else:
            trainer.train()

        metrics = trainer.evaluate()
        f1 = float(metrics.get("eval_macro_f1", np.nan))
        fold_scores.append(f1)

        row = {"fold": fold, "seed": fold_seed}
        row.update({k: v for k, v in metrics.items() if isinstance(v, (int, float, np.number))})
        fold_metrics_rows.append(row)

        print(f"\n[FOLD {fold}] eval_macro_f1 = {f1}")

        dev_ds = FusionTextDataset(
            dev_df, rob_tokenizer, mpnet_tok,
            text_col=cfg["text_col"],
            label_col=None,
            max_length=cfg["max_length"],
        )

        dev_logits = trainer.predict(dev_ds).predictions
        dev_probs = softmax(dev_logits, axis=-1)
        dev_probs_sum += dev_probs
        completed_folds += 1

    if completed_folds == 0:
        raise RuntimeError("No folds were run. Check cv_start_fold/cv_end_fold/cv_folds in config.yaml.")


    dev_probs_avg = dev_probs_sum / float(completed_folds)
    dev_pred = np.argmax(dev_probs_avg, axis=-1)

    mean_f1 = float(np.nanmean(fold_scores))
    std_f1 = float(np.nanstd(fold_scores))
    print("\n=== CV SUMMARY ===")
    print("Folds run:", list(range(cfg["cv_start_fold"], cfg["cv_end_fold"] + 1)))
    print("Fold macro-F1:", fold_scores)
    print(f"Mean macro-F1: {mean_f1:.6f} | Std: {std_f1:.6f} | Completed folds: {completed_folds}")

    metrics_path = Path(cfg["outputs_dir"]) / "cv_fold_metrics.csv"
    pd.DataFrame(fold_metrics_rows).to_csv(metrics_path, index=False)
    print("Saved:", metrics_path)

    out_path = Path(cfg["outputs_dir"]) / "eng_dev_predictions_cv_ensemble.csv"
    out = pd.DataFrame({cfg["id_col"]: dev_df[cfg["id_col"]].values, "pred_polarization": dev_pred})
    out.to_csv(out_path, index=False)
    print("Saved dev predictions to:", out_path)




if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

/kaggle/input/augment/eng_aug.csv
Train label counts:
 polarization
0    1842
1    1057
Name: count, dtype: int64
Test label counts:
 polarization
0    205
1    118
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...


=== Confusion Matrix (rows=true, cols=pred) ===
[[191  14]
 [ 10 108]]

=== Classification report ===
              precision    recall  f1-score   support

           0     0.9502    0.9317    0.9409       205
           1     0.8852    0.9153    0.9000       118

    accuracy                         0.9257       323
   macro avg     0.9177    0.9235    0.9204       323
weighted avg     0.9265    0.9257    0.9259       323


[FOLD 4] eval_macro_f1 = 0.9204433497536946


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Precision 0,Recall 0,F1 0,Precision 1,Recall 1,F1 1
100,2.675800,0.621639,0.653251,0.441107,0.646688,1.000000,0.785441,1.000000,0.050847,0.096774
200,1.877100,0.491404,0.767802,0.745303,0.803738,0.839024,0.821002,0.697248,0.644068,0.669604
300,2.043500,0.377230,0.851393,0.840336,0.886700,0.878049,0.882353,0.791667,0.805085,0.798319
400,1.622300,0.371068,0.832817,0.827444,0.931429,0.795122,0.857895,0.716216,0.898305,0.796992
500,1.433600,0.314209,0.873065,0.865204,0.918367,0.878049,0.897756,0.803150,0.864407,0.832653
600,1.316500,0.308991,0.869969,0.862559,0.922280,0.868293,0.894472,0.792308,0.872881,0.830645
700,1.385800,0.292221,0.882353,0.874867,0.923858,0.887805,0.905473,0.817460,0.872881,0.844262
800,1.165200,0.290097,0.885449,0.877963,0.924242,0.892683,0.908189,0.824000,0.872881,0.847737



=== Confusion Matrix (rows=true, cols=pred) ===
[[205   0]
 [112   6]]

=== Classification report ===
              precision    recall  f1-score   support

           0     0.6467    1.0000    0.7854       205
           1     1.0000    0.0508    0.0968       118

    accuracy                         0.6533       323
   macro avg     0.8233    0.5254    0.4411       323
weighted avg     0.7758    0.6533    0.5339       323


=== Confusion Matrix (rows=true, cols=pred) ===
[[172  33]
 [ 42  76]]

=== Classification report ===
              precision    recall  f1-score   support

           0     0.8037    0.8390    0.8210       205
           1     0.6972    0.6441    0.6696       118

    accuracy                         0.7678       323
   macro avg     0.7505    0.7415    0.7453       323
weighted avg     0.7648    0.7678    0.7657       323


=== Confusion Matrix (rows=true, cols=pred) ===
[[180  25]
 [ 23  95]]

=== Classification report ===
              precision    recall  f1


=== Confusion Matrix (rows=true, cols=pred) ===
[[183  22]
 [ 15 103]]

=== Classification report ===
              precision    recall  f1-score   support

           0     0.9242    0.8927    0.9082       205
           1     0.8240    0.8729    0.8477       118

    accuracy                         0.8854       323
   macro avg     0.8741    0.8828    0.8780       323
weighted avg     0.8876    0.8854    0.8861       323


[FOLD 5] eval_macro_f1 = 0.8779626055611719

=== CV SUMMARY ===
Folds run: [1, 2, 3, 4, 5]
Fold macro-F1: [0.8843025705857563, 0.904031555760463, 0.9090283210765138, 0.9204433497536946, 0.8779626055611719]
Mean macro-F1: 0.899154 | Std: 0.015774 | Completed folds: 5
Saved: /kaggle/working/outputs/cv_fold_metrics.csv
Saved dev predictions to: /kaggle/working/outputs/eng_dev_predictions_cv_ensemble.csv
